In [ ]:
import pandas as pd
import geopandas as gpd
import folium
import numpy as np

# 1. INGESTA DE DATOS ESPACIALES (IDS)
# IMPORTANTE: Cambia 'ids_cdmx.shp' por el nombre exacto de tu archivo shapefile
gdf_ids = gpd.read_file('../data/raw/ids_cdmx.shp')

# 2. INGESTA Y LIMPIEZA DEL CENSO (INEGI)
# CORRECCIÓN: Leemos directamente desde la fila 1 sin saltar metadatos
df_censo = pd.read_excel('../data/raw/cpv2020.xlsx')

# Limpieza de confidencialidad del INEGI (los asteriscos)
df_censo.replace({'*': np.nan, 'N/D': np.nan}, inplace=True)

# Seleccionamos variables (ahora sí las va a encontrar)
columnas_interes = ['ENTIDAD', 'MUN', 'LOC', 'AGEB', 'POBTOT', 'GRAPROES']
df_censo = df_censo[columnas_interes]

# Reconstrucción del CVEGEO (13 dígitos) para asegurar el cruce exacto
df_censo['CVEGEO'] = (
    df_censo['ENTIDAD'].astype(str).str.zfill(2) + 
    df_censo['MUN'].astype(str).str.zfill(3) + 
    df_censo['LOC'].astype(str).str.zfill(4) + 
    df_censo['AGEB'].astype(str).str.zfill(4)
)

# Convertimos la escolaridad a valores numéricos
df_censo['GRAPROES'] = pd.to_numeric(df_censo['GRAPROES'], errors='coerce')

# 3. FUSIÓN ESPACIAL (MERGE)
# Estandarizamos el nombre de la llave en el Censo para que sea idéntico al Shapefile
df_censo.rename(columns={'CVEGEO': 'cvegeo'}, inplace=True)

# Ahora el cruce es perfectamente simétrico
gdf_master = gdf_ids.merge(df_censo[['cvegeo', 'POBTOT', 'GRAPROES']], on='cvegeo', how='inner')

# 4. REPROYECCIÓN Y OPTIMIZACIÓN ESPACIAL (CRÍTICO)
gdf_master = gdf_master.to_crs(epsg=4326)

# Algoritmo de simplificación de Douglas-Peucker. 
# 0.0005 grados son aprox 50 metros. Reduce el HTML de 225MB a ~10MB.
gdf_master['geometry'] = gdf_master.simplify(0.0005)

# 5. RENDERIZADO DEL MAPA CLASE MUNDIAL (FOLIUM)
mapa_cdmx = folium.Map(location=[19.4326, -99.1332], zoom_start=11, tiles='CartoDB dark_matter')

folium.Choropleth(
    geo_data=gdf_master,
    name='Grado Promedio de Escolaridad',
    data=gdf_master,
    columns=['cvegeo', 'GRAPROES'],
    key_on='feature.properties.cvegeo',
    fill_color='YlGnBu',
    fill_opacity=0.7,
    line_opacity=0.2,
    nan_fill_color='black',
    legend_name='Años de Escolaridad Promedio'
).add_to(mapa_cdmx)

mapa_cdmx.save('../docs/index.html')
print("Mapa renderizado, optimizado y exportado con éxito.")

Mapa renderizado y exportado a la carpeta docs con éxito.


In [5]:
print(gdf_ids.columns.tolist())

['cvegeo', 'cve_ent', 'cve_mun', 'cve_loc', 'cve_ageb', 'pobtotal', 'pobres_tot', 'ids_ccevj', 'ids_csj', 'ids_caej', 'ids_ctelj', 'ids_cbdj', 'ids_rei', 'ids_cassi', 'ids_casi', 'e_idsm', 'ids', 'geometry']
